In [1]:
import pandas as pd

# 0. Open the data (the original listings file)
date_columns = ["last_scraped", "price_quote_checkin_date", "price_quote_checkout_date", "calendar_last_scraped", "first_review", "last_review"]
final = pd.read_csv("listings_original.csv", low_memory=False, parse_dates=date_columns)

# In the csv file the price comes as text (for example "$76.31"), so it is changed to a number
final["price"] = final["price"].str.replace("$", "", regex=False).str.replace(",", "", regex=False).astype(float)
print("Original rows:", len(final))

# 1. Host filter: only hosts with 1 or 2 listings
final = final[final["calculated_host_listings_count"] <= 2].copy()
print("Rows after the host filter:", len(final))

# The amenities text has codes for special characters: they are changed to the real characters
final["amenities"] = final["amenities"].str.replace("\\u2019", "’", regex=False)
final["amenities"] = final["amenities"].str.replace("\\u2013", "–", regex=False)

# Special spaces in the codes (\u00a0 and \u202f) are changed to normal spaces, and repeated spaces are reduced to one
final["amenities"] = final["amenities"].str.replace("\\u00a0", " ", regex=False)
final["amenities"] = final["amenities"].str.replace("\\u202f", " ", regex=False)
final["amenities"] = final["amenities"].str.replace("  ", " ", regex=False)
final["amenities"] = final["amenities"].str.replace("  ", " ", regex=False)

# One property type has a wrong accent encoding in the original file
final["property_type"] = final["property_type"].replace({"Shepherd‚Äôs hut": "Shepherd’s hut"})

# 2. Price: missing and extreme values are only flagged
final["price_missing"] = 0
final.loc[final["price"].isna(), "price_missing"] = 1

price_upper = final["price"].quantile(0.99)
final["price_outlier"] = 0
final.loc[(final["price"] < 20) | (final["price"] > price_upper), "price_outlier"] = 1

# 3. Bedrooms: blank values filled with the median of similar listings (same room type and guest capacity)
final["bedrooms_original"] = final["bedrooms"]
final["bedrooms_imputed"] = 0
final.loc[final["bedrooms"].isna(), "bedrooms_imputed"] = 1

group_median = final.groupby(["room_type", "accommodates"])["bedrooms"].transform("median")
group_count = final.groupby(["room_type", "accommodates"])["bedrooms"].transform("count")
room_median = final.groupby("room_type")["bedrooms"].transform("median")
group_median.loc[group_count < 10] = room_median
final["bedrooms"] = final["bedrooms"].fillna(group_median)

# 4. Neighbourhood: neighborhoods with fewer than 10 listings are merged into the nearest one
neighbourhood_map = {
    "Angeles Crest": "Sierra Madre",
    "Arleta": "Panorama City",
    "Artesia": "Cerritos",
    "Avocado Heights": "South El Monte",
    "Bell Gardens": "Bell",
    "Chatsworth Reservoir": "Chatsworth",
    "Citrus": "Azusa",
    "Desert View Highlands": "Palmdale",
    "East Compton": "Paramount",
    "East Whittier": "La Mirada",
    "Elizabeth Lake": "Quartz Hill",
    "Elysian Park": "Echo Park",
    "Florence": "Florence-Firestone",
    "Green Meadows": "Broadway-Manchester",
    "Green Valley": "Agua Dulce",
    "Griffith Park": "Los Feliz",
    "Harvard Park": "Chesterfield Square",
    "Hasley Canyon": "Castaic",
    "Hawaiian Gardens": "Cerritos",
    "Hidden Hills": "Calabasas",
    "Huntington Park": "Florence-Firestone",
    "Industry": "Walnut",
    "Irwindale": "Duarte",
    "Lake Hughes": "Quartz Hill",
    "Lake Los Angeles": "Southeast Antelope Valley",
    "Lake View Terrace": "Pacoima",
    "Leona Valley": "Quartz Hill",
    "Lopez/Kagel Canyons": "Pacoima",
    "Maywood": "Bell",
    "North El Monte": "Mayflower Village",
    "North Whittier": "West Whittier-Los Nietos",
    "Northeast Antelope Valley": "Southeast Antelope Valley",
    "Northwest Antelope Valley": "Lancaster",
    "Rancho Dominguez": "Compton",
    "Ridge Route": "Castaic",
    "Rolling Hills": "Rolling Hills Estates",
    "San Marino": "San Pasqual",
    "Santa Fe Springs": "South Whittier",
    "Sepulveda Basin": "Encino",
    "South Park": "Vermont-Slauson",
    "South San Jose Hills": "Rowland Heights",
    "Sun Village": "Southeast Antelope Valley",
    "Sunland": "Tujunga",
    "Universal City": "Toluca Lake",
    "Val Verde": "Castaic",
    "Vincent": "Azusa",
    "Walnut Park": "South Gate",
    "Watts": "South Gate",
    "West Compton": "Willowbrook",
    "West Puente Valley": "Baldwin Park",
    "Westlake Village": "Agoura Hills",
}
final["neighbourhood_model"] = final["neighbourhood_cleansed"].replace(neighbourhood_map)

# 5. Property type: rare types (fewer than 100 listings) are grouped by room type
type_count = final["property_type"].map(final["property_type"].value_counts())
final["property_type_model"] = final["property_type"]
final.loc[(type_count < 100) & (final["room_type"] == "Entire home/apt"), "property_type_model"] = "Unique entire place"
final.loc[(type_count < 100) & (final["room_type"] == "Private room"), "property_type_model"] = "Unique private room"
final.loc[(type_count < 100) & (final["room_type"] != "Entire home/apt") & (final["room_type"] != "Private room"), "property_type_model"] = "Shared room"

# 6. Amenities: which amenities are kept (those in between 5% and 90% of the listings)
amenity_list = final["amenities"].str.lower().str.replace("’", "'", regex=False).str.strip('[]"').str.split('", "')
amenity_items = amenity_list.explode().str.strip()
amenity_items = amenity_items[amenity_items != ""]
amenity_share = amenity_items.value_counts() / len(final)
kept_amenities = amenity_share[(amenity_share >= 0.05) & (amenity_share <= 0.90)]
print("Different amenities:", len(amenity_share))
print("Amenities kept (5% to 90%):", len(kept_amenities))

# Each kept amenity belongs to one group: one point per amenity in each group
amenity_groups = {
  "am_kitchen_score": [
    "Kitchen",
    "Dishes and silverware",
    "Microwave",
    "Cooking basics",
    "Refrigerator",
    "Freezer",
    "Dining table",
    "Wine glasses",
    "Toaster",
    "Dishwasher",
    "Hot water kettle",
    "Coffee",
    "Oven",
    "Coffee maker",
    "Stove",
    "Baking sheet",
    "Blender",
    "Mini fridge",
    "Rice maker",
    "Gas stove",
    "Coffee maker: Keurig coffee machine",
    "Coffee maker: drip coffee maker",
    "Stainless steel oven",
    "Children’s dinnerware",
    "Trash compactor",
  ],
  "am_bath_score": [
    "Hot water",
    "Essentials",
    "Hair dryer",
    "Shampoo",
    "Cleaning products",
    "Shower gel",
    "Body soap",
    "Conditioner",
    "Bidet",
  ],
  "am_security_score": [
    "Carbon monoxide alarm",
    "Fire extinguisher",
    "First aid kit",
    "Exterior security cameras on property",
    "Keypad",
    "Lockbox",
    "Smart lock",
    "Lock on bedroom door",
    "Safe",
  ],
  "am_bedroom_score": [
    "Hangers",
    "Bed linens",
    "Iron",
    "Extra pillows and blankets",
    "Room-darkening shades",
    "Clothing storage",
    "Drying rack for clothing",
    "Clothing storage: closet",
    "Clothing storage: closet and dresser",
    "Pack ’n play/Travel crib",
  ],
  "am_outdoor_score": [
    "Outdoor dining area",
    "Outdoor furniture",
    "BBQ grill",
    "Private patio or balcony",
    "Barbecue utensils",
    "Fire pit",
    "Private backyard – Fully fenced",
    "Backyard",
    "Patio or balcony",
    "Sun loungers",
    "Beach essentials",
    "Shared patio or balcony",
  ],
  "am_entertainment_work_score": [
    "TV",
    "Dedicated workspace",
    "Books and reading material",
    "Board games",
    "Ethernet connection",
    "Exercise equipment",
    "TV with standard cable",
    "Sound system",
  ],
  "am_access_services_score": [
    "Self check-in",
    "Private entrance",
    "Luggage dropoff allowed",
    "Host greets you",
    "Cleaning available during stay",
    "Elevator",
  ],
  "am_views_score": [
    "Garden view",
    "Mountain view",
    "Beach access",
    "City skyline view",
  ],
  "am_other_score": [
    "Long term stays allowed",
    "Single level home",
    "Ceiling fan",
    "Laundromat nearby",
    "Indoor fireplace",
    "Portable fans",
    "Children’s books and toys",
  ],
}

for score_name, items in amenity_groups.items():
    final[score_name] = 0
    for item in items:
        final[score_name] = final[score_name] + final["amenities"].str.contains('"' + item + '"', regex=False)

# Brand and model variants (for example "Pantene shampoo" or "Samsung refrigerator") count as the amenity itself, only when the listing does not already have it
brand_items = {
  "am_kitchen_score": ["gas stove", "stove", "oven", "refrigerator"],
  "am_bath_score": ["shampoo", "conditioner", "body soap"],
  "am_entertainment_work_score": ["sound system", "tv"],
}

amenities_lower = final["amenities"].str.lower()
for score_name, generics in brand_items.items():
    for generic in generics:
        found = amenity_items[amenity_items.str.endswith(" " + generic)]
        found = found[~found.isin(kept_amenities.index)]
        if generic == "stove":
            found = found[~found.str.endswith("gas stove")]
        has_generic = amenities_lower.str.contains('"' + generic + '"', regex=False)
        extra = pd.Series(0, index=final.index)
        extra.loc[found.index.unique()] = 1
        final[score_name] = final[score_name] + extra.where(~has_generic, 0)

# 7. Amenities: 0/1 columns (1 = the listing has at least one amenity of the list)
has_groups = {
  "has_ac": [
    "Air conditioning",
    "Central air conditioning",
    "AC - split type ductless system",
  ],
  "has_heating": [
    "Heating",
    "Central heating",
    "Heating - split type ductless system",
    "Portable heater",
  ],
  "has_laundry": [
    "Washer",
    "Dryer",
    "Free dryer – In unit",
    "Free washer – In unit",
  ],
  "has_parking": [
    "Free parking on premises",
    "Free street parking",
  ],
  "has_pool": [
    "Pool",
  ],
  "has_hot_tub": [
    "Hot tub",
  ],
  "has_bathtub": [
    "Bathtub",
  ],
  "pets_allowed": [
    "Pets allowed",
  ],
}

for has_name, items in has_groups.items():
    final[has_name] = 0
    for item in items:
        final.loc[final["amenities"].str.contains('"' + item + '"', regex=False), has_name] = 1

# The 0/1 columns also count the amenities that are written in other ways (for example "Private pool" or "Washer – In unit")
has_words = {
  "has_ac": ["air conditioning", "window ac", "ac - "],
  "has_heating": ["heating", "portable heater"],
  "has_laundry": ["washer", "dryer"],
  "has_parking": ["parking", "garage"],
  "has_pool": ["pool"],
  "has_hot_tub": ["hot tub"],
  "has_bathtub": ["bathtub"],
  "pets_allowed": ["pets allowed"],
}
# Words that look similar but are not the amenity
skip_words = ["dishwasher", "hair dryer", "pool view", "pool table", "pool toys", "whirlpool", "whirpool", "refrigerator", "oven", "stove", "in garage"]

for has_name, words in has_words.items():
    for word in words:
        found = amenity_items[amenity_items.str.contains(word, regex=False)]
        for skip in skip_words:
            found = found[~found.str.contains(skip, regex=False)]
        final.loc[found.index, has_name] = 1

all_items = []
for items in list(amenity_groups.values()) + list(has_groups.values()):
    all_items = all_items + items
print("Amenities in the groups:", len(all_items))
all_items_lower = [item.lower().replace("’", "'").strip() for item in all_items]
print("Same as the kept amenities:", set(all_items_lower) == set(kept_amenities.index))

final["amenities_count"] = final["amenities"].str.count('", "') + 1
final["amenities_empty"] = 0
final.loc[final["amenities"] == "[]", "amenities_empty"] = 1
final.loc[final["amenities_empty"] == 1, "amenities_count"] = 0

# 8. Minimum nights: not modified, only flagged
final["long_stay"] = 0
final.loc[final["minimum_nights"] > 30, "long_stay"] = 1
final["minimum_nights_missing"] = 0
final.loc[final["minimum_nights"].isna(), "minimum_nights_missing"] = 1

# 9. Keep the final columns
final = final[["id", "latitude", "longitude", "room_type", "calculated_host_listings_count", "price", "price_missing", "price_outlier", "bedrooms", "bedrooms_original", "bedrooms_imputed", "accommodates", "neighbourhood_cleansed", "neighbourhood_model", "property_type", "property_type_model", "am_kitchen_score", "am_bath_score", "am_security_score", "am_bedroom_score", "am_outdoor_score", "am_entertainment_work_score", "am_access_services_score", "am_views_score", "am_other_score", "has_ac", "has_heating", "has_laundry", "has_parking", "has_pool", "has_hot_tub", "has_bathtub", "pets_allowed", "amenities_count", "amenities_empty", "minimum_nights", "long_stay", "minimum_nights_missing"]]
print("Final rows and columns:", final.shape)

# 10. Export

# 11. Data dictionary: one row per variable of the clean table
data_dictionary = pd.DataFrame({
    "variable": [
        "id",
        "latitude",
        "longitude",
        "room_type",
        "calculated_host_listings_count",
        "price",
        "price_missing",
        "price_outlier",
        "bedrooms",
        "bedrooms_original",
        "bedrooms_imputed",
        "accommodates",
        "neighbourhood_cleansed",
        "neighbourhood_model",
        "property_type",
        "property_type_model",
        "am_kitchen_score",
        "am_bath_score",
        "am_security_score",
        "am_bedroom_score",
        "am_outdoor_score",
        "am_entertainment_work_score",
        "am_access_services_score",
        "am_views_score",
        "am_other_score",
        "has_ac",
        "has_heating",
        "has_laundry",
        "has_parking",
        "has_pool",
        "has_hot_tub",
        "has_bathtub",
        "pets_allowed",
        "amenities_count",
        "amenities_empty",
        "minimum_nights",
        "long_stay",
        "minimum_nights_missing",
    ],
    "description": [
        "Listing identifier",
        "Listing latitude",
        "Listing longitude",
        "Rental type (entire home, private room, etc.)",
        "Number of listings of the host",
        "Nightly price in USD",
        "Listing has no price",
        "Extreme price",
        "Number of bedrooms",
        "Bedrooms before filling blank values",
        "Bedrooms value was filled in",
        "Guest capacity",
        "Original neighborhood",
        "Neighborhood used by the model",
        "Original property type",
        "Property type used by the model",
        "Kitchen amenities score",
        "Bathroom amenities score",
        "Security amenities score",
        "Bedroom amenities score",
        "Outdoor amenities score",
        "Entertainment and work amenities score",
        "Access and services amenities score",
        "Views amenities score",
        "Other amenities score",
        "Has air conditioning",
        "Has heating",
        "Has laundry",
        "Has parking",
        "Has pool",
        "Has hot tub",
        "Has bathtub",
        "Pets allowed",
        "Total number of amenities",
        "Amenities list is empty",
        "Minimum nights required",
        "Asks for more than 30 nights",
        "Minimum nights is blank",
    ],
    "expected_type": [
        "categorical",
        "numeric",
        "numeric",
        "categorical",
        "numeric",
        "numeric",
        "categorical (0/1)",
        "categorical (0/1)",
        "numeric",
        "numeric",
        "categorical (0/1)",
        "numeric",
        "categorical",
        "categorical",
        "categorical",
        "categorical",
        "numeric",
        "numeric",
        "numeric",
        "numeric",
        "numeric",
        "numeric",
        "numeric",
        "numeric",
        "numeric",
        "categorical (0/1)",
        "categorical (0/1)",
        "categorical (0/1)",
        "categorical (0/1)",
        "categorical (0/1)",
        "categorical (0/1)",
        "categorical (0/1)",
        "categorical (0/1)",
        "numeric",
        "categorical (0/1)",
        "numeric",
        "categorical (0/1)",
        "categorical (0/1)",
    ],
    "important_note": [
        "Original data",
        "Used only to merge small neighborhoods; not a model variable",
        "Used only to merge small neighborhoods; not a model variable",
        "Helper only: fills bedrooms and groups property types; not a model variable",
        "Host filter: only hosts with 1 or 2 listings kept (20,541 properties)",
        "Dependent variable. $ and commas removed. 4,237 blank values not filled in",
        "1 = no price (4,237). Left out when training",
        "1 = below $20 or above $2,157 (99th percentile) (169). Left out when training",
        "58 blank values filled with the median of similar listings",
        "Keeps the original data",
        "1 = filled in (58)",
        "No changes",
        "Not modified",
        "Neighborhoods with fewer than 10 listings merged into the nearest one (263 to 212)",
        "Not modified",
        "62 rare types grouped into Unique entire place, Unique private room and Shared room (80 to 21)",
        "One point per amenity in the group",
        "One point per amenity in the group",
        "One point per amenity in the group",
        "One point per amenity in the group",
        "One point per amenity in the group",
        "One point per amenity in the group",
        "One point per amenity in the group",
        "One point per amenity in the group",
        "One point per amenity in the group",
        "1 = yes, 0 = no",
        "1 = yes, 0 = no",
        "1 = yes, 0 = no",
        "1 = yes, 0 = no",
        "1 = yes, 0 = no",
        "1 = yes, 0 = no",
        "1 = yes, 0 = no",
        "1 = yes, 0 = no",
        "Support column",
        "1 = empty list (20)",
        "Not modified; not a model variable. 11 blank values",
        "1 = long stay (1,808 in total). Left out when training",
        "1 = blank (11). Kept in training",
    ],
})

# Both tables are saved in the same Excel file, each one in its own sheet
with pd.ExcelWriter("listings_clean.xlsx") as writer:
    data_dictionary.to_excel(writer, sheet_name="Data dictionary", index=False)
    final.to_excel(writer, sheet_name="Clean data", index=False)

Original rows: 43781
Rows after the host filter: 20541
Different amenities: 7187
Amenities kept (5% to 90%): 107
Amenities in the groups: 107
Same as the kept amenities: True
Final rows and columns: (20541, 38)
